In [1]:
import csv
import json
import os
import re
import pandas as pd
import prosodic

In [2]:
with open("hopkins.txt", "r", encoding="utf-8") as file:
    book = file.read()

# took the following from word cloud example in week 10
start = book.index("*** START OF")
end = book.index("*** END OF")
text = book[start:end]

In [3]:
titles = re.findall(r'_(\d+.*?)_', text, re.DOTALL) #googled some regex for this
print(len(titles))

71


In [4]:
print(titles[0])

1
For a Picture of
St. Dorothea


In [5]:
poems = []
for i in range(70):
    poem_start = text.index(titles[i]) #taken from prof's example in week 10
    poem_end = text.index(titles[i+1])
    if poem_end is not None: #taken from prof's work on my first try
        poem = text[poem_start:poem_end]
    else: 
        poem = text[poem_start:end]
    pt = poem.replace("_\n\n", "")
    pt = pt.strip("\n\n\n")
    pt = pt.strip("\n\n\n_")
    poem_object = {
        "title": titles[i],
        "poem_text": pt.replace(titles[i], "")
    }
    poems.append(poem_object) 

In [6]:
print(poems[0]) #checking it worked

{'title': '1\nFor a Picture of\nSt. Dorothea', 'poem_text': "I BEAR a basket lined with grass;\nI am so light, I am so fair,\nThat men must wonder as I pass\nAnd at the basket that I bear,\nWhere in a newly-drawn green litter\nSweet flowers I carry,--sweets for bitter.\n\nLilies I shew you, lilies none,\nNone in Caesar's gardens blow,--\nAnd a quince in hand,--not one\nIs set upon your boughs below;\nNot set, because their buds not spring;\nSpring not, 'cause world is wintering.\n\nBut these were found in the East and South\nWhere Winter is the clime forgot.--\nThe dewdrop on the larkspur's mouth\nO should it then be quenchèd not?\nIn starry water-meads they drew\nThese drops: which be they? stars or dew?\n\nHad she a quince in hand? Yet gaze:\nRather it is the sizing moon.\nLo, linked heavens with milky ways!\nThat was her larkspur row.--So soon?\nSphered so fast, sweet soul?--We see\nNor fruit, nor flowers, nor Dorothy."}


In [7]:
df = pd.DataFrame(poems) #creating a dataframe to make it easier to navigate

In [8]:
df.head()

,title,poem_text
0,1\nFor a Picture of\nSt. Dorothea,I BEAR a basket lined with grass;\nI am so lig...
1,2\nHeaven--Haven\nA nun takes the veil,I HAVE desired to go\n Where springs no...
2,3\nThe Habit of Perfection,"ELECTED Silence, sing to me\nAnd beat upon my ..."
3,4\nTHE WRECK\nOF THE DEUTSCHLAND,To the\nhappy memory of five Francis...
4,5\nPenmaen Pool,_For the Visitors' Book at the InnWHO long for...


In [9]:
all_parsed_poems = []
valid_poems = df["poem_text"].dropna() #I was getting an error saying the string can't evaluate to none, so I'm dropping null values
valid_poems = valid_poems[valid_poems.str.strip().astype(bool)] #google said to add this extra step removing empty strings: "" 
for vp in valid_poems[:69]:
    if vp is not None:
        parsable_poem = prosodic.Text(vp)
        parsed = parsable_poem.parse()
        # add some lines to compute the integers for the counts 
        poem_info = {
            "feet_per_line" : parsable_poem.line_scheme, #this is a dict
            "syll_per_line" : parsable_poem.syllable_scheme, #this is a dict
            "line_count" : len(parsable_poem.lines),
            "stanza_count" : len(parsable_poem.stanzas),
            "parsed_text" : parsable_poem.Text.txt,
            "summary" : parsable_poem.summary()
            }
        all_parsed_poems.append(poem_info)


[18.60s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 line(s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=22 sylls); falling back to linepart parsing
[0.04s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 (s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=22 sylls)
[0.23s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 line(s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=21 sylls); falling back to linepart parsing
[1.43s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 (s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=21 sylls)
[9.83s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 line(s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=25 sylls); falling back to linepart parsing
[0.62s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 (s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=25 sylls)
[8.67s] prosodic.parsing.vectorized.parse_batch_from_df(): 9 line(s) exceed MAX_SYLL_IN

In [36]:
parsed_df = pd.DataFrame(all_parsed_poems)

,feet_per_line,syll_per_line,line_count,stanza_count,parsed_text,summary
0,"{'combo': (4, 4, 4, 4, 4, 4, 4, 3), 'diff': 9}","{'combo': (8,), 'diff': 9}",24,4,I BEAR a basket lined with grass;\nI am so lig...,#st #ln parse rhyme #feet #...
1,"{'combo': (3, 2, 5, 3), 'diff': 1}","{'combo': (6, 4, 10, 6, 6, 4, 10, 8), 'diff': 1}",8,2,I HAVE desired to go\n Where springs not f...,#st #ln parse rhyme #feet ...
2,"{'combo': (4,), 'diff': 6}","{'combo': (8,), 'diff': 16}",29,8,"ELECTED Silence, sing to me\nAnd beat upon my ...",#st #ln parse rhyme #feet #...
3,"{'combo': (4,), 'diff': 820}","{'combo': (9,), 'diff': 1096}",334,37,To the\nhappy memory of five Franciscan Nuns\n...,#st #ln parse rhyme #f...
4,"{'combo': (4,), 'diff': 10}","{'combo': (8,), 'diff': 28}",40,10,_For the Visitors' Book at the InnWHO long for...,#st #ln parse rhyme #...
...,...,...,...,...,...,...
62,"{'combo': (2, 4, 6, 8, 4), 'diff': 1}","{'combo': (6, 8, 16, 16, 8), 'diff': 3}",5,3,How all's to one thing wrought!\n\n_See facsim...,#st #ln parse rhyme #f...
63,"{'combo': (4, 3, 3, 4, 4, 4, 4, 4), 'diff': 1}","{'combo': (8,), 'diff': 0}",8,1,'The child is father to the man.'\nHow can he ...,#st #ln parse rhyme #feet #s...
64,"{'combo': (5,), 'diff': 4}","{'combo': (10,), 'diff': 19}",15,1,"THE shepherd's brow fronting forked lightning,...",#st #ln parse rhyme #feet ...
65,"{'combo': (5,), 'diff': 8}","{'combo': (10,), 'diff': 7}",11,2,"MORTAL my mate, bearing my rock-a-heart\nWarm ...",#st #ln parse rhyme #feet ...


In [37]:
parsed_df["line_count"].describe()

count     67.000000
mean      50.388060
std      121.775745
min        4.000000
25%       14.000000
50%       15.000000
75%       25.000000
max      815.000000
Name: line_count, dtype: float64

In [38]:
parsed_df["stanza_count"].describe()

count     67.000000
mean       6.970149
std       18.907246
min        1.000000
25%        2.000000
50%        2.000000
75%        4.000000
max      143.000000
Name: stanza_count, dtype: float64

In [184]:
all_parsed_lines = []
for vp in valid_poems[:69]:
    if vp is not None:
        parsable_poem = prosodic.Text(vp)
        parsable_poem.parse() 
        if parsable_poem.lines is not None:
            for l in parsable_poem.lines:
                  if l is not None and getattr(l, "best_parse", None) is not None and l.best_parse.feet is not None:
                      line_info = {
                          "parsed_text": l.txt,
                          "feet_in_line": len(l.best_parse.feet),
                          "stress": l.best_parse.stress_str,
                          "foot_type": l.best_parse.foot_type,
                    }
                      all_parsed_lines.append(line_info)

[154.49s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 line(s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=22 sylls); falling back to linepart parsing
[0.05s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 (s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=22 sylls)
[0.09s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 line(s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=21 sylls); falling back to linepart parsing
[1.65s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 (s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=21 sylls)
[2.72s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 line(s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=25 sylls); falling back to linepart parsing
[0.62s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 (s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=25 sylls)
[6.58s] prosodic.parsing.vectorized.parse_batch_from_df(): 9 line(s) exceed MAX_SYLL_I

In [186]:
parsed_line_df = pd.DataFrame(all_parsed_lines)
parsed_line_df

,parsed_text,feet_in_line,stress,foot_type
0,I BEAR a basket lined with grass;,4,-+-+-+-+,iambic
1,"\nI am so light, I am so fair,",4,+--+---+,iambic
2,\nThat men must wonder as I pass,4,-+-+---+,iambic
3,"\nAnd at the basket that I bear,",4,-+-+-+-+,iambic
4,\nWhere in a newly-drawn green litter,5,-+-+-+++-,iambic
...,...,...,...,...
3197,\nAnd then that last and shortest . . .,4,-+-+-+-,iambic
3198,"STRIKE, churl; hurl, cheerless wind, then; hel...",5,++++-+-+--+,iambic
3199,\nMay's beauty massacre and wispèd wild clouds...,6,++-+---++++,iambic
3200,"\nOut on the giant air; tell Summer No,",5,---+-+++-+,iambic


In [187]:
poem_numbers = []
for index, row in parsed_line_df.iterrows():
    line = row["parsed_text"]
    poem_number = None
    for i, p in enumerate(parsed_df["parsed_text"]):
        if line in p:
            poem_number = i
            break
    poem_numbers.append(poem_number)
    
parsed_line_df["poem_number"] = poem_numbers

In [188]:
parsed_line_df

,parsed_text,feet_in_line,stress,foot_type,poem_number
0,I BEAR a basket lined with grass;,4,-+-+-+-+,iambic,0.0
1,"\nI am so light, I am so fair,",4,+--+---+,iambic,0.0
2,\nThat men must wonder as I pass,4,-+-+---+,iambic,0.0
3,"\nAnd at the basket that I bear,",4,-+-+-+-+,iambic,0.0
4,\nWhere in a newly-drawn green litter,5,-+-+-+++-,iambic,0.0
...,...,...,...,...,...
3197,\nAnd then that last and shortest . . .,4,-+-+-+-,iambic,65.0
3198,"STRIKE, churl; hurl, cheerless wind, then; hel...",5,++++-+-+--+,iambic,66.0
3199,\nMay's beauty massacre and wispèd wild clouds...,6,++-+---++++,iambic,66.0
3200,"\nOut on the giant air; tell Summer No,",5,---+-+++-+,iambic,66.0


In [189]:
parsed_line_df["feet_in_line"].describe()

count    3202.000000
mean        4.351031
std         1.469192
min         1.000000
25%         3.000000
50%         4.000000
75%         5.000000
max         9.000000
Name: feet_in_line, dtype: float64

In [190]:
parsed_line_df["feet_in_line"].value_counts()

feet_in_line
5    928
3    682
4    662
6    437
2    168
7    165
1    114
8     43
9      3
Name: count, dtype: int64

In [191]:
parsed_line_df["foot_type"].value_counts() # none ar spondaic!

foot_type
iambic       1943
trochaic      906
anapestic     198
dactylic      150
                5
Name: count, dtype: int64

In [193]:
parsed_line_df["stress"].value_counts()

stress
-+-+-+         129
-+-+-+-+-+     104
-+-+-+-+        68
+-              52
+-+             47
              ... 
++--+-+++-+      1
++++-+-+--+      1
++-+---++++      1
---+-+++-+       1
+++-+-+-+++      1
Name: count, Length: 962, dtype: int64

In [194]:
even_spondee_count = 0
spondaic_lines = []
poems_with_spondee = []
for index, row in parsed_line_df.iterrows():
    if row["feet_in_line"] % 2 == 0:
        if "+++" in row["stress"]:
            even_spondee_count = even_spondee_count + 1
            spondaic_lines.append(row["parsed_text"])
            poems_with_spondee.append(row["poem_number"]),

In [195]:
odd_spondee_count = 0
potential_spondee = []
potentially_spondaic_lines = []
poems_with_potential_spondee = []
for index, row in parsed_line_df.iterrows():
    if row["feet_in_line"]:
        if "+++" in row["stress"]:
            odd_spondee_count = odd_spondee_count + 1
            potentially_spondaic_lines.append(row["parsed_text"])
            poems_with_potential_spondee.append(row["poem_number"])

In [196]:
surely_spondaic = round((len(spondaic_lines)/3201) * 100)
print(surely_spondaic)

12


In [197]:
maybe_spondaic = round((len(potentially_spondaic_lines)/3201) * 100)
print(maybe_spondaic)

30


In [198]:
spondee_max = round(maybe_spondaic + surely_spondaic)
print(spondee_max)

42


In [199]:
unique_poems_with_spondee = sorted(list(dict.fromkeys(poems_with_spondee)))
cleaned_poems_with_spondee = [x for x in unique_poems_with_spondee if str(x) != "nan"] # found here: https://stackoverflow.com/questions/21011777/how-can-i-remove-nan-from-list-python-numpy
numeric_poems_with_spondee = [int(x) for x in cleaned_poems_with_spondee] # found here: https://stackoverflow.com/questions/35946190/removing-decimal-point-from-list
percent_poems_with_spondee = round((len(numeric_poems_with_spondee)/68)*100)
print(percent_poems_with_spondee)

84


In [183]:
#for index, row in parsed_line_df.iterrows():
   #if row["poem_number"] == 0:
        #print(row["parsed_text"])

### Issues to note in final summary
- The parsing doesn't work with any white space
- The parsing has a limit of 18 syllables per line, and some lines are longer, so it falls back to a "line-part"

### Ideas - parsing poems
- Manually select a few poems as case studies (early career, mid-career, late career / at least one sprung rhythm example / one with white space?) -- maybe just for visual examples.
  
### Ideas - analysis
- count line length per poem over time and on average
- count feet per line over time and on average
- count the most common patterns (iambic, trochaic, spondaic, dactyllic, anapestic)